# LeetCode 808 — Soup Servings bằng Python

Link đề: https://leetcode.com/problems/soup-servings/ · Độ khó: **Medium** · Chủ đề: Math, Dynamic Programming, Probability and Statistics

Skeleton LeetCode:

```python
class Solution:
    def soupServings(self, n: int) -> float:
        ...
```

Ghi chú: kết quả là **số thực** và đề chấp nhận sai số `10^-5`. Vì vậy notebook dùng `fractions.Fraction` để tính **xác suất đúng tuyệt đối** làm chuẩn, rồi so sánh các cài đặt dùng số thực **có sai số** (`abs(a - b) <= 1e-9`), và kiểm tra thống kê bằng mô phỏng Monte Carlo với RNG tự dựng có seed.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hai nồi súp A và B cùng `n` mL; mỗi lượt chọn **ngẫu nhiên đều** một trong 4 thao tác múc `(A, B)` = `(100, 0)`, `(75, 25)`, `(50, 50)`, `(25, 75)`; dừng khi một nồi cạn; tính **P(A cạn trước) + ½·P(cùng cạn)**.
2. Nắm các công cụ Python: đệ quy có nhớ (`functools.cache`/`lru_cache`), quy hoạch động dưới lên trên bảng 2 chiều, `fractions.Fraction` (xác suất chính xác), so sánh số thực với dung sai, `RecursionError`, và mô phỏng ngẫu nhiên có seed.
3. Đi từ **đệ quy thuần tính theo mL (mũ)** → **nén đơn vị 25 mL + nhớ kết quả + cắt cụt khi `n` lớn** → **DP bảng dưới lên** → **lan truyền phân phối xác suất theo từng lượt (forward)**.
4. Hiểu mấu chốt: mọi thao tác đều là bội của `25`, nên chỉ cần xét `N = ⌈n/25⌉` **đơn vị**; trạng thái `(i, j)` là số đơn vị còn lại, chỉ có `N²` trạng thái. Và khi `n` đủ lớn (`> 4800`) đáp án gần `1` đến mức `1 − 10^-5`, nên trả `1`.
5. Tránh các lỗi kinh điển: chia sàn thay vì chia **lên** (ceil), nhầm giá trị cơ sở (hai nồi cùng cạn là `0.5`), kiểm tra điều kiện sai thứ tự, quên cắt cụt cho `n` lớn (tràn đệ quy), và so sánh số thực bằng `==`.

## 2. Đề bài & ví dụ

Có hai nồi súp **A** và **B**, mỗi nồi ban đầu có `n` mL. Mỗi lượt, một trong bốn thao tác múc sau được chọn **ngẫu nhiên**, mỗi thao tác xác suất `0,25`, **độc lập** với các lượt trước:

1. múc 100 mL từ A và 0 mL từ B;
2. múc 75 mL từ A và 25 mL từ B;
3. múc 50 mL từ A và 50 mL từ B;
4. múc 25 mL từ A và 75 mL từ B.

Chú ý:

- Không có thao tác múc 0 mL từ A và 100 mL từ B.
- Lượng múc từ A và B diễn ra **đồng thời** trong một lượt.
- Nếu thao tác đòi múc **nhiều hơn** lượng còn lại của một nồi thì múc hết phần còn lại của nồi đó.

Quá trình **dừng ngay** sau lượt mà **một trong hai nồi** hết súp.

Trả về **xác suất nồi A hết trước nồi B, cộng với một nửa xác suất cả hai nồi cùng hết trong cùng một lượt**. Đáp án sai khác không quá `10^-5` so với đáp án thật được chấp nhận.

**Ví dụ 1:** `n = 50` → `0,62500`. Thao tác 1, 2: A hết trước. Thao tác 3: A và B hết cùng lúc. Thao tác 4: B hết trước. Xác suất `0,25 × (1 + 1 + 0,5 + 0) = 0,625`.

**Ví dụ 2:** `n = 100` → `0,71875`.

**Ràng buộc:** `0 <= n <= 10^9`.

In [1]:
import sys
import math
import time
import random
from fractions import Fraction
from functools import cache, lru_cache
from typing import List

OPS = [(100, 0), (75, 25), (50, 50), (25, 75)]     # (múc từ A, múc từ B) tính bằng mL
print("4 thao tác:", OPS, "| mỗi thao tác xác suất 1/4")
print("trung bình mỗi lượt múc từ A:", sum(a for a, _ in OPS) / 4, "mL, từ B:", sum(b for _, b in OPS) / 4, "mL")

4 thao tác: [(100, 0), (75, 25), (50, 50), (25, 75)] | mỗi thao tác xác suất 1/4
trung bình mỗi lượt múc từ A: 62.5 mL, từ B: 37.5 mL


## 3. Kiến thức nền

### 3.1 Xác suất của một chuỗi lựa chọn độc lập

Nếu mỗi lượt có 4 kết quả đồng khả năng (`1/4`) và các lượt độc lập, thì xác suất một **chuỗi** cụ thể gồm `t` lượt là `(1/4)^t`. "Xác suất của một biến cố" là tổng xác suất các chuỗi dẫn tới biến cố đó. Ví dụ `n = 50`: thao tác 1, 2 → A hết trước (1 lượt); 3 → hòa; 4 → B hết trước:
`P(A trước) = 2/4`, `P(hòa) = 1/4`, `P(B trước) = 1/4` → đáp án `P(A trước) + ½ P(hòa) = 0,5 + 0,125 = 0,625`.

In [2]:
p_a, p_tie, p_b = Fraction(2, 4), Fraction(1, 4), Fraction(1, 4)
print("P(A trước) =", p_a, "| P(hòa) =", p_tie, "| P(B trước) =", p_b, "| tổng =", p_a + p_tie + p_b)
print("đáp án n=50:", p_a + p_tie / 2, "=", float(p_a + p_tie / 2))

P(A trước) = 1/2 | P(hòa) = 1/4 | P(B trước) = 1/4 | tổng = 1
đáp án n=50: 5/8 = 0.625


### 3.2 Công thức truy hồi xác suất (luật xác suất toàn phần)

Gọi `f(a, b)` là đáp án khi A còn `a` mL, B còn `b` mL. Theo thao tác chọn lượt đầu:

```
f(a, b) = 1/4 · [ f(a−100, b) + f(a−75, b−25) + f(a−50, b−50) + f(a−25, b−75) ]
```

với **giá trị cơ sở** (khi trừ ra ≤ 0, nồi đó đã hết):

- `a ≤ 0` và `b ≤ 0`: cùng hết → `0,5`.
- `a ≤ 0` và `b > 0`: A hết trước → `1`.
- `a > 0` và `b ≤ 0`: B hết trước → `0`.

Trường hợp múc nhiều hơn lượng còn lại **cũng coi là hết**, nên trừ ra số âm vẫn đúng ý "múc hết phần còn lại" (ta chỉ cần biết nồi đã hết hay chưa).

### 3.3 Đệ quy có nhớ (`functools.cache`)

Cùng một trạng thái `(a, b)` xuất hiện từ nhiều đường nên tính lại rất lãng phí. `@cache` lưu kết quả mỗi cặp đối số đã gặp. Số trạng thái khác nhau là hữu hạn nên thời gian = số trạng thái × công việc mỗi trạng thái.

In [3]:
calls = [0]


@cache
def paths(i, j):
    """Số đường đi từ (i, j) về (0, 0) bằng bước (-1,0) hoặc (0,-1): minh hoạ tác dụng của cache."""
    calls[0] += 1
    if i == 0 or j == 0:
        return 1
    return paths(i - 1, j) + paths(i, j - 1)


print(paths(14, 14), "| số lần tính thật:", calls[0], "(không cache sẽ hơn 40 triệu lần)")

40116600 | số lần tính thật: 224 (không cache sẽ hơn 40 triệu lần)


### 3.4 Chia lên (ceil) bằng số nguyên

Mọi lượng múc là bội của `25`, nên ta đổi đơn vị `25 mL → 1`: `N = ⌈n / 25⌉`. Trong Python không có hàm ceil cho `//`, nhưng `(n + 24) // 25` làm đúng việc đó (hoặc `-(-n // 25)`). **Chia sàn** `n // 25` sai khi `n` không chia hết `25`.

In [4]:
for n in (0, 1, 24, 25, 26, 49, 50, 51):
    print(f"n={n:3d}: n//25 = {n // 25}, ceil = {(n + 24) // 25}, math.ceil = {math.ceil(n / 25)}")

n=  0: n//25 = 0, ceil = 0, math.ceil = 0
n=  1: n//25 = 0, ceil = 1, math.ceil = 1
n= 24: n//25 = 0, ceil = 1, math.ceil = 1
n= 25: n//25 = 1, ceil = 1, math.ceil = 1
n= 26: n//25 = 1, ceil = 2, math.ceil = 2
n= 49: n//25 = 1, ceil = 2, math.ceil = 2
n= 50: n//25 = 2, ceil = 2, math.ceil = 2
n= 51: n//25 = 2, ceil = 3, math.ceil = 3


### 3.5 `Fraction` và so sánh số thực có dung sai

`Fraction` cho xác suất **chính xác** (không sai số). Với số thực `float`, hai phép tính tương đương toán học có thể khác nhau ở chữ số cuối, nên **không dùng `==`**; dùng `abs(a - b) <= eps` (hoặc `math.isclose`). Đề chấp nhận sai số `10^-5`.

In [5]:
print(0.1 + 0.2 == 0.3, abs((0.1 + 0.2) - 0.3) < 1e-12, math.isclose(0.1 + 0.2, 0.3))
x = Fraction(1, 4) * (Fraction(1) + Fraction(1) + Fraction(1, 2) + Fraction(0))
print(x, float(x), x == Fraction(5, 8))

False True True
5/8 0.625 True


### 3.6 Đệ quy sâu và `RecursionError`

Python giới hạn độ sâu đệ quy mặc định khoảng `1000`. Đệ quy có nhớ ở bài này đi sâu tối đa `N` tầng (mỗi bước A mất ít nhất 1 đơn vị). Với `n = 10^9`, `N = 4·10^7` → chắc chắn tràn. Đó là lý do cần **cắt cụt** (trả `1` khi `n` lớn) và là gợi ý rằng đáp án tiến dần tới `1`.

In [6]:
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())


def depth(k):
    return 0 if k == 0 else 1 + depth(k - 1)


try:
    depth(10 ** 5)
except RecursionError:
    print("depth(100000) -> RecursionError")

giới hạn đệ quy mặc định: 1000
depth(100000) -> RecursionError


## 4. Ý tưởng nền tảng / trực giác

Tính tay `n = 50`. Đổi sang đơn vị 25 mL: `N = 2`; các thao tác thành `(4,0), (3,1), (2,2), (1,3)`. Trạng thái đầu `(2, 2)`:

- `(4,0)`: A còn `2−4 = −2 ≤ 0`, B còn `2 > 0` → A hết trước → `1`.
- `(3,1)`: A còn `−1 ≤ 0`, B còn `1 > 0` → `1`.
- `(2,2)`: A còn `0`, B còn `0` → cùng hết → `0,5`.
- `(1,3)`: A còn `1 > 0`, B còn `−1 ≤ 0` → B hết trước → `0`.

`f(2,2) = ¼·(1 + 1 + 0,5 + 0) = 0,625` ✓ (khớp ví dụ 1).

**Nhận xét:** trung bình mỗi lượt A mất `62,5 mL` còn B chỉ mất `37,5 mL`, nên **A hết trước B** có khuynh hướng rõ rệt. Khi `n` rất lớn, theo luật số lớn, A gần như chắc chắn hết trước; đáp án tiến tới `1`. Điều này cho phép **cắt cụt** (xem mục 10).

Ta tính các giá trị đầu của dãy để có cảm nhận:

In [7]:
@cache
def f_exact(i, j):
    """Đáp án chính xác (Fraction) khi A còn i đơn vị, B còn j đơn vị (1 đơn vị = 25 mL)."""
    if i <= 0 and j <= 0:
        return Fraction(1, 2)
    if i <= 0:
        return Fraction(1)
    if j <= 0:
        return Fraction(0)
    return Fraction(1, 4) * (f_exact(i - 4, j) + f_exact(i - 3, j - 1) + f_exact(i - 2, j - 2) + f_exact(i - 1, j - 3))


print("N   n(mL)   f(N,N)")
for N in (1, 2, 3, 4, 5, 8, 12, 20):
    v = f_exact(N, N)
    print(f"{N:2d}  {N * 25:5d}   {v!s:>12} = {float(v):.6f}")

N   n(mL)   f(N,N)
 1     25            5/8 = 0.625000
 2     50            5/8 = 0.625000
 3     75          21/32 = 0.656250
 4    100          23/32 = 0.718750
 5    125         95/128 = 0.742188
 8    200          51/64 = 0.796875
12    300      6981/8192 = 0.852173
20    500   1921713/2097152 = 0.916344


## 5. Cách 1 — Brute Force: đệ quy thuần tính theo mL

**Ý tưởng:** viết đúng công thức truy hồi trên **mL thật** (không nén đơn vị, không nhớ), bằng `Fraction` để chính xác tuyệt đối. Mỗi lượt gọi 4 nhánh; độ sâu tối đa `⌈n/25⌉`, nên số lời gọi tới `4^N`.

In [8]:
def soup_brute(n):
    """Đệ quy thuần theo mL, không memo; chỉ dùng cho n nhỏ. Trả Fraction."""
    calls = [0]

    def go(a, b):
        calls[0] += 1
        if a <= 0 and b <= 0:
            return Fraction(1, 2)
        if a <= 0:
            return Fraction(1)
        if b <= 0:
            return Fraction(0)
        return Fraction(1, 4) * sum(go(a - da, b - db) for da, db in OPS)

    res = go(n, n)
    soup_brute.calls = calls[0]
    return res


for n in (0, 1, 25, 50, 100, 150, 200):
    r = soup_brute(n)
    print(f"n={n:4d}: {r!s:>10} = {float(r):.6f}  ({soup_brute.calls:>8,d} lời gọi)")
assert soup_brute(50) == Fraction(5, 8) and soup_brute(100) == Fraction(23, 32)

n=   0:        1/2 = 0.500000  (       1 lời gọi)
n=   1:        5/8 = 0.625000  (       5 lời gọi)
n=  25:        5/8 = 0.625000  (       5 lời gọi)
n=  50:        5/8 = 0.625000  (       5 lời gọi)
n= 100:      23/32 = 0.718750  (      17 lời gọi)
n= 150:     97/128 = 0.757812  (      57 lời gọi)
n= 200:      51/64 = 0.796875  (     193 lời gọi)


**Độ phức tạp:** thời gian `O(4^N)` với `N = ⌈n/25⌉` (hàng triệu lời gọi cho `n = 250`); bộ nhớ `O(N)` (độ sâu đệ quy). Dùng được làm oracle cho `n ≤ ~250`.

**Nút thắt:** cùng trạng thái `(a, b)` được tính lại rất nhiều lần (nhiều đường dẫn tới cùng `(a, b)`), và ta tính trên mL (nhiều trạng thái hơn cần thiết vì mọi lượng đều bội của 25).

## 6. Ý tưởng thuật toán / bước đột phá

Ba bước then chốt:

1. **Nén đơn vị:** mọi thao tác là bội của 25 mL, nên đổi `n` thành `N = ⌈n / 25⌉` đơn vị; các thao tác thành `(4,0), (3,1), (2,2), (1,3)`. Trạng thái `(i, j)` với `0 ≤ i, j ≤ N`: chỉ `N²` trạng thái. (Chứng minh phép nén ở mục 10.)
2. **Nhớ kết quả (memoization / DP):** mỗi trạng thái tính một lần: `O(N²)` thời gian.
3. **Cắt cụt khi `n` lớn:** đáp án `f(n)` không giảm và tiến tới `1`; với `n > 4800` (tức `N > 192`) ta có `1 − f < 10^-5`, nên trả `1`. Do đó `N ≤ 192` và chỉ cần `~37 000` trạng thái.

Ngoài ra có thể nhìn quá trình dạng **chuỗi Markov hấp thụ**: lan truyền phân phối xác suất theo từng lượt (cách forward, mục 7).

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** đệ quy có nhớ + nén đơn vị + cắt cụt, **Cách 3** DP bảng dưới lên, **Cách 4** lan truyền phân phối theo từng lượt (forward).

### Cách 2 — Đệ quy có nhớ, nén đơn vị, cắt cụt

**Giải thích từng đoạn:**

1. Nếu `n > 4800` trả `1.0`.
2. `N = (n + 24) // 25`; gọi `dfs(N, N)`.
3. `dfs(i, j)`: các trường hợp cơ sở như mục 3.2 (`i ≤ 0 and j ≤ 0 → 0.5`, `i ≤ 0 → 1`, `j ≤ 0 → 0`); còn lại `0.25 × (dfs(i−4, j) + dfs(i−3, j−1) + dfs(i−2, j−2) + dfs(i−1, j−3))`, kèm `@cache`.

Thứ tự kiểm tra cơ sở **quan trọng**: phải kiểm tra "cả hai ≤ 0" trước, nếu không `i ≤ 0` sẽ chặn mất trường hợp hòa.

In [9]:
def soup_memo(n):
    if n > 4800:
        return 1.0

    @cache
    def dfs(i, j):
        if i <= 0 and j <= 0:
            return 0.5
        if i <= 0:
            return 1.0
        if j <= 0:
            return 0.0
        return 0.25 * (dfs(i - 4, j) + dfs(i - 3, j - 1) + dfs(i - 2, j - 2) + dfs(i - 1, j - 3))

    return dfs((n + 24) // 25, (n + 24) // 25)


print(soup_memo(50), soup_memo(100), soup_memo(0), soup_memo(10 ** 9))
assert soup_memo(50) == 0.625 and soup_memo(100) == 0.71875

0.625 0.71875 0.5 1.0


In [10]:
# Code nộp LeetCode cho Cách 2
from functools import cache


class SolutionMemo:
    def soupServings(self, n: int) -> float:
        @cache
        def dfs(i: int, j: int) -> float:
            if i <= 0 and j <= 0:
                return 0.5
            if i <= 0:
                return 1
            if j <= 0:
                return 0
            return 0.25 * (dfs(i - 4, j) + dfs(i - 3, j - 1) + dfs(i - 2, j - 2) + dfs(i - 1, j - 3))

        return 1 if n > 4800 else dfs((n + 24) // 25, (n + 24) // 25)

**Độ phức tạp:** `O(N²)` với `N ≤ 192` → `~37 000` trạng thái × 4 nhánh. Độ sâu đệ quy `≤ N ≤ 192` nên an toàn. Bộ nhớ `O(N²)`.

### Cách 3 — DP bảng dưới lên

**Ý tưởng:** thay đệ quy bằng bảng `f[i][j]` (`1 ≤ i, j ≤ N`), điền theo thứ tự tăng của `i, j`. Vì `f[i][j]` chỉ phụ thuộc `f[i−4][j], f[i−3][j−1], f[i−2][j−2], f[i−1][j−3]` (chỉ số nhỏ hơn), điền theo `i` tăng, `j` tăng là đủ. Dùng hàm phụ `get(i, j)` trả giá trị cơ sở khi `i ≤ 0` hoặc `j ≤ 0`.

In [11]:
def soup_table(n):
    if n > 4800:
        return 1.0
    N = (n + 24) // 25
    if N == 0:
        return 0.5
    f = [[0.0] * (N + 1) for _ in range(N + 1)]

    def get(i, j):
        if i <= 0 and j <= 0:
            return 0.5
        if i <= 0:
            return 1.0
        if j <= 0:
            return 0.0
        return f[i][j]

    for i in range(1, N + 1):
        for j in range(1, N + 1):
            f[i][j] = 0.25 * (get(i - 4, j) + get(i - 3, j - 1) + get(i - 2, j - 2) + get(i - 1, j - 3))
    return f[N][N]


print(soup_table(50), soup_table(100), soup_table(0), soup_table(10 ** 9))
assert soup_table(50) == 0.625 and soup_table(100) == 0.71875 and soup_table(0) == 0.5

0.625 0.71875 0.5 1.0


In [12]:
# Code nộp LeetCode cho Cách 3
class SolutionTable:
    def soupServings(self, n: int) -> float:
        if n > 4800:
            return 1.0
        N = (n + 24) // 25
        if N == 0:
            return 0.5
        f = [[0.0] * (N + 1) for _ in range(N + 1)]

        def get(i: int, j: int) -> float:
            if i <= 0 and j <= 0:
                return 0.5
            if i <= 0:
                return 1.0
            if j <= 0:
                return 0.0
            return f[i][j]

        for i in range(1, N + 1):
            for j in range(1, N + 1):
                f[i][j] = 0.25 * (get(i - 4, j) + get(i - 3, j - 1) + get(i - 2, j - 2) + get(i - 1, j - 3))
        return f[N][N]

**Độ phức tạp:** `O(N²)` thời gian và bộ nhớ, không đệ quy (không lo `RecursionError`). Bảng cũng cho đáp án của **mọi** `N' ≤ N` một lúc: `f[N'][N']`.

### Cách 4 — Lan truyền phân phối xác suất theo từng lượt (forward)

**Ý tưởng khác:** nhìn quá trình như chuỗi Markov. Giữ một **bản đồ** `dist[(i, j)] = xác suất đang ở trạng thái (i, j) (chưa dừng)` sau `t` lượt. Mỗi lượt, từ mỗi trạng thái đi 4 hướng với xác suất `1/4`: nếu trạng thái mới đã dừng thì **cộng vào đáp án**: A hết trước (`i' ≤ 0, j' > 0`) cộng `p`; cùng hết (`i' ≤ 0, j' ≤ 0`) cộng `p/2`; B hết trước cộng `0`; còn lại chuyển sang `dist` mới. Lặp tới khi không còn trạng thái chưa dừng (A mất ít nhất 1 đơn vị mỗi lượt nên không quá `N` lượt).

Cách này tính **xuôi** (từ trạng thái đầu) thay vì **ngược** (từ trạng thái cuối): kết quả giống nhau nhưng đường đi tính toán khác hẳn — rất tiện để cross-check.

In [13]:
def soup_forward(n, exact=False):
    if n > 4800 and not exact:
        return 1.0
    N = (n + 24) // 25
    one = Fraction(1) if exact else 1.0
    quarter = Fraction(1, 4) if exact else 0.25
    half = Fraction(1, 2) if exact else 0.5
    if N == 0:
        return half
    dist = {(N, N): one}
    answer = one * 0
    while dist:
        new = {}
        for (i, j), p in dist.items():
            q = p * quarter
            for da, db in ((4, 0), (3, 1), (2, 2), (1, 3)):
                i2, j2 = i - da, j - db
                if i2 <= 0 and j2 <= 0:
                    answer += q * half
                elif i2 <= 0:
                    answer += q
                elif j2 <= 0:
                    pass                           # B hết trước: không cộng
                else:
                    new[(i2, j2)] = new.get((i2, j2), 0) + q
        dist = new
    return answer


print(soup_forward(50), soup_forward(100), soup_forward(0), soup_forward(10 ** 9))
assert soup_forward(50) == 0.625 and soup_forward(100) == 0.71875
print("chính xác:", soup_forward(100, exact=True), "=", Fraction(23, 32))
assert soup_forward(100, exact=True) == Fraction(23, 32)

0.625 0.71875 0.5 1.0
chính xác: 23/32 = 23/32


In [14]:
# Code nộp LeetCode cho Cách 4
class SolutionForward:
    def soupServings(self, n: int) -> float:
        if n > 4800:
            return 1.0
        N = (n + 24) // 25
        if N == 0:
            return 0.5
        dist = {(N, N): 1.0}
        ans = 0.0
        while dist:
            new = {}
            for (i, j), p in dist.items():
                q = p * 0.25
                for da, db in ((4, 0), (3, 1), (2, 2), (1, 3)):
                    i2, j2 = i - da, j - db
                    if i2 <= 0 and j2 <= 0:
                        ans += q * 0.5
                    elif i2 <= 0:
                        ans += q
                    elif j2 > 0:
                        new[(i2, j2)] = new.get((i2, j2), 0.0) + q
            dist = new
        return ans

**Độ phức tạp:** mỗi lượt xử lý tối đa `N²` trạng thái × 4, tối đa `N` lượt → `O(N³)` thời gian trong trường hợp xấu (thực tế trạng thái đạt được ít hơn `N²`); bộ nhớ `O(N²)`. Chậm hơn Cách 2/3 nhưng cho **bằng chứng độc lập**.

## 8. TRACE từng bước

**Trace Cách 3** với `n = 100` → `N = 4`. Bảng `f[i][j]` (điền theo `i`, rồi `j`). Dùng giá trị cơ sở: `get(i ≤ 0, j ≤ 0) = 0,5`; `get(i ≤ 0, j > 0) = 1`; `get(i > 0, j ≤ 0) = 0`.

Một số ô tính tay:

- `f[1][1] = ¼·(get(−3,1) + get(−2,0) + get(−1,−1) + get(0,−2)) = ¼·(1 + 0,5 + 0,5 + 0,5) = 0,625`.
- `f[2][2] = ¼·(get(−2,2) + get(−1,1) + get(0,0) + get(1,−1)) = ¼·(1 + 1 + 0,5 + 0) = 0,625`.
- `f[4][4] = ¼·(get(0,4) + get(1,3) + get(2,2) + get(3,1)) = ¼·(1 + f[1][3] + f[2][2] + f[3][1])`.

Hàm trace in từng ô của bảng và các giá trị thành phần:

In [15]:
def trace_table(n, show_all=False):
    N = (n + 24) // 25
    print(f"n = {n} mL -> N = {N} đơn vị (25 mL)")
    f = [[0.0] * (N + 1) for _ in range(N + 1)]

    def get(i, j):
        if i <= 0 and j <= 0:
            return 0.5
        if i <= 0:
            return 1.0
        if j <= 0:
            return 0.0
        return f[i][j]

    for i in range(1, N + 1):
        for j in range(1, N + 1):
            parts = [get(i - 4, j), get(i - 3, j - 1), get(i - 2, j - 2), get(i - 1, j - 3)]
            f[i][j] = 0.25 * sum(parts)
            if show_all or (i == j):
                print(f"  f[{i}][{j}] = 1/4 * ({' + '.join(f'{p:g}' for p in parts)}) = {f[i][j]:.6f}")
    print(f"=> đáp án f[{N}][{N}] = {f[N][N]}")
    return f[N][N]


assert trace_table(100) == 0.71875
print()
assert trace_table(50) == 0.625

n = 100 mL -> N = 4 đơn vị (25 mL)
  f[1][1] = 1/4 * (1 + 0.5 + 0.5 + 0.5) = 0.625000
  f[2][2] = 1/4 * (1 + 1 + 0.5 + 0) = 0.625000
  f[3][3] = 1/4 * (1 + 1 + 0.625 + 0) = 0.656250
  f[4][4] = 1/4 * (1 + 0.875 + 0.625 + 0.375) = 0.718750
=> đáp án f[4][4] = 0.71875

n = 50 mL -> N = 2 đơn vị (25 mL)
  f[1][1] = 1/4 * (1 + 0.5 + 0.5 + 0.5) = 0.625000
  f[2][2] = 1/4 * (1 + 1 + 0.5 + 0) = 0.625000
=> đáp án f[2][2] = 0.625


**Bảng `f[i][j]` đầy đủ cho `N = 4`** (hàng `i`, cột `j`) — in bằng hàm dưới; ô `f[4][4] = 0,71875` là đáp án của ví dụ 2:

In [16]:
def full_table(N):
    f = [[0.0] * (N + 1) for _ in range(N + 1)]

    def get(i, j):
        if i <= 0 and j <= 0:
            return 0.5
        if i <= 0:
            return 1.0
        if j <= 0:
            return 0.0
        return f[i][j]

    for i in range(1, N + 1):
        for j in range(1, N + 1):
            f[i][j] = 0.25 * (get(i - 4, j) + get(i - 3, j - 1) + get(i - 2, j - 2) + get(i - 1, j - 3))
    return f


tbl = full_table(4)
print("      " + "".join(f"j={j:<8d}" for j in range(1, 5)))
for i in range(1, 5):
    print(f"i={i}  " + "".join(f"{tbl[i][j]:<10.5f}" for j in range(1, 5)))

      j=1       j=2       j=3       j=4       
i=1  0.62500   0.75000   0.87500   1.00000   
i=2  0.50000   0.62500   0.75000   0.90625   
i=3  0.37500   0.50000   0.65625   0.81250   
i=4  0.25000   0.40625   0.56250   0.71875   


**Trace Cách 4 (forward)** với `n = 50` (`N = 2`): lượt 1 từ `(2,2)`: bốn nhánh đều dừng ngay (`(4,0)`: A hết → `+0,25`; `(3,1)`: A hết → `+0,25`; `(2,2)`: hòa → `+0,125`; `(1,3)`: B hết → `+0`). Tổng `0,625`. Hàm trace in từng lượt:

In [17]:
def trace_forward(n):
    N = (n + 24) // 25
    dist = {(N, N): Fraction(1)}
    answer = Fraction(0)
    turn = 0
    print(f"n = {n}, N = {N}")
    while dist:
        turn += 1
        new = {}
        gained = Fraction(0)
        for (i, j), p in sorted(dist.items()):
            for da, db in ((4, 0), (3, 1), (2, 2), (1, 3)):
                i2, j2, q = i - da, j - db, p / 4
                if i2 <= 0 and j2 <= 0:
                    gained += q / 2
                elif i2 <= 0:
                    gained += q
                elif j2 > 0:
                    new[(i2, j2)] = new.get((i2, j2), 0) + q
        answer += gained
        print(f"lượt {turn}: đang chờ {len(dist)} trạng thái, cộng {gained} vào đáp án -> {answer} | trạng thái còn lại: {dict(sorted(new.items()))}")
        dist = new
    print("=> đáp án", answer, "=", float(answer))
    return answer


assert trace_forward(50) == Fraction(5, 8)
print()
assert trace_forward(100) == Fraction(23, 32)

n = 50, N = 2
lượt 1: đang chờ 1 trạng thái, cộng 5/8 vào đáp án -> 5/8 | trạng thái còn lại: {}
=> đáp án 5/8 = 0.625

n = 100, N = 4
lượt 1: đang chờ 1 trạng thái, cộng 1/4 vào đáp án -> 1/4 | trạng thái còn lại: {(1, 3): Fraction(1, 4), (2, 2): Fraction(1, 4), (3, 1): Fraction(1, 4)}
lượt 2: đang chờ 3 trạng thái, cộng 15/32 vào đáp án -> 23/32 | trạng thái còn lại: {}
=> đáp án 23/32 = 0.71875


## 9. Code nộp LeetCode

Chọn **đệ quy có nhớ + nén đơn vị + cắt cụt** (ngắn gọn nhất). Giữ nguyên `class Solution`, `soupServings(self, n)`:

In [18]:
from functools import cache


class Solution:
    def soupServings(self, n: int) -> float:
        @cache
        def dfs(i: int, j: int) -> float:
            # i, j: số đơn vị 25 mL còn lại của A và B
            if i <= 0 and j <= 0:       # cùng hết trong một lượt
                return 0.5
            if i <= 0:                  # A hết trước
                return 1
            if j <= 0:                  # B hết trước
                return 0
            return 0.25 * (
                dfs(i - 4, j)           # múc 100 / 0
                + dfs(i - 3, j - 1)     # múc 75 / 25
                + dfs(i - 2, j - 2)     # múc 50 / 50
                + dfs(i - 1, j - 3)     # múc 25 / 75
            )

        # n > 4800: đáp án cách 1 không quá 1e-5, trả luôn 1
        return 1 if n > 4800 else dfs((n + 24) // 25, (n + 24) // 25)


print(Solution().soupServings(50), Solution().soupServings(100))   # 0.625 0.71875
assert Solution().soupServings(50) == 0.625 and Solution().soupServings(100) == 0.71875

0.625 0.71875


## 10. Vì sao đúng

**(a) Nén đơn vị đúng.** Gọi `a` là lượng mL còn lại; mọi thao tác trừ một bội của 25. Sau `m` đơn vị đã múc từ nồi, nồi **đã hết** khi `a − 25m ≤ 0 ⇔ m ≥ ⌈a/25⌉`. Vậy hành vi (hết hay chưa) chỉ phụ thuộc `⌈a/25⌉`. Hai trạng thái `(a, b)`, `(a', b')` với cùng `⌈a/25⌉ = ⌈a'/25⌉`, `⌈b/25⌉ = ⌈b'/25⌉` có cùng đáp án. Do đó thay `n` bằng `N = ⌈n/25⌉` đơn vị là đúng. Ta kiểm chứng tuyệt đối bằng máy (mục 12D): đệ quy brute theo mL cho kết quả **bằng chính xác** `Fraction` với bản đơn vị.

**(b) Truy hồi đúng.** Theo luật xác suất toàn phần: `f(a, b) = Σ_{op} ¼ · f(a − da, b − db)`; khi một nồi cạn, quá trình dừng nên `f` là giá trị cơ sở (`1`, `0` hoặc `0,5`). Quá trình kết thúc: mỗi lượt A mất ít nhất 25 mL (thao tác có `da ≥ 25`), nên sau tối đa `N` lượt A cạn; do đó đệ quy dừng và `f` được xác định duy nhất.

**(c) Cắt cụt `n > 4800`.** Mỗi lượt A mất trung bình `62,5 mL`, B mất `37,5 mL`; chênh lệch `25 mL` mỗi lượt nghiêng về A. Sau `t` lượt, hiệu `(A đã mất) − (B đã mất)` là tổng của `t` biến độc lập có kỳ vọng `+25` và bị chặn trong `[−75, 100]`. Để B hết trước (hoặc hòa) thì hiệu này phải rất lệch so với kỳ vọng, và theo bất đẳng thức Hoeffding xác suất đó giảm theo hàm mũ khi `n` tăng. Do đó `P(B hết trước)` và `P(hòa)` tiến về `0` rất nhanh và đáp án `f(n) → 1`. Ta không dựa vào lý thuyết suông mà **đo** bằng số học chính xác: `1 − f(N, N)` với `N = 192` (`n = 4800`) cỡ `5·10^-6 < 10^-5`, và `f(N, N)` **không giảm** theo `N` (kiểm chứng đến `N = 600`), nên mọi `n > 4800` đều có `1 − f < 10^-5` (mục 12D).

**(d) Chính xác số học.** Với `float`, mỗi phép cộng/nhân có sai số tương đối `~10^-16`; qua tối đa `~N·4` phép, sai số cộng dồn cỡ `10^-13`, rất nhỏ so với dung sai `10^-5`. Ta kiểm chứng bằng cách so với `Fraction` ở mục 12D.

## 11. Time / Space Complexity

Đặt `N = ⌈n/25⌉ ≤ 192` sau khi cắt cụt (với `n > 4800` trả ngay `O(1)`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Đệ quy thuần theo mL | `O(4^N)` | `O(N)` | chỉ làm oracle (`n ≤ 250`) |
| 2. Đệ quy nhớ + cắt cụt | `O(N²)` | `O(N²)` | **đáp án chính**; độ sâu `≤ N` |
| 3. DP bảng | `O(N²)` | `O(N²)` | không đệ quy; cho cả `f(N', N')` với `N' ≤ N` |
| 4. Forward | `O(N³)` xấu nhất | `O(N²)` | cross-check độc lập |

`N²` tối đa `192² ≈ 37 000` trạng thái nên chạy tức thì. Đo thực tế:

In [19]:
def bench(fn, n, repeat=5):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(n)
        best = min(best, time.perf_counter() - t0)
    return best


for n in (100, 1000, 4800, 4801, 10 ** 9):
    row = [f"memo {bench(soup_memo, n) * 1000:8.3f} ms", f"table {bench(soup_table, n) * 1000:8.3f} ms", f"forward {bench(soup_forward, n, 2) * 1000:9.3f} ms"]
    print(f"n = {n:10d}: " + " | ".join(row))
t0 = time.perf_counter()
soup_brute(250)
print(f"brute n=250: {(time.perf_counter() - t0) * 1000:.0f} ms ({soup_brute.calls:,d} lời gọi)")

n =        100: memo    0.007 ms | table    0.010 ms | forward     0.005 ms
n =       1000: memo    0.256 ms | table    0.920 ms | forward     0.449 ms


n =       4800: memo    6.385 ms | table   20.485 ms | forward     5.738 ms
n =       4801: memo    0.000 ms | table    0.000 ms | forward     0.000 ms
n = 1000000000: memo    0.000 ms | table    0.000 ms | forward     0.000 ms
brute n=250: 1 ms (665 lời gọi)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ: cài đặt sai phải bị bắt, (D) **so sánh chính xác bằng `Fraction`** và có dung sai cho `float`, kiểm chứng cắt cụt, (E) mô phỏng Monte Carlo với RNG tự dựng có seed (kiểm thống kê), (F) giới hạn lớn nhất.

Quy ước so sánh số thực: `abs(a − b) <= 1e-9` giữa các cài đặt, và `<= 1e-5` (dung sai của đề) so với đáp án chính xác.

### A. Ví dụ của đề

In [20]:
IMPLS = {
    "memo": soup_memo,
    "table": soup_table,
    "forward": soup_forward,
    "Solution": lambda n: Solution().soupServings(n),
    "SolutionMemo": lambda n: SolutionMemo().soupServings(n),
    "SolutionTable": lambda n: SolutionTable().soupServings(n),
    "SolutionForward": lambda n: SolutionForward().soupServings(n),
}
EPS = 1e-9


def close(a, b, eps=EPS):
    return abs(a - b) <= eps


def check(n, want, eps=EPS, names=None):
    for name in (names or IMPLS):
        got = IMPLS[name](n)
        assert close(got, float(want), eps), (name, n, got, float(want))


check(50, Fraction(5, 8))
check(100, Fraction(23, 32))
assert soup_brute(50) == Fraction(5, 8) and soup_brute(100) == Fraction(23, 32)
print("ví dụ 1, 2: cả", len(IMPLS), "cài đặt và brute đúng (n=50 -> 0.625, n=100 -> 0.71875)")

ví dụ 1, 2: cả 7 cài đặt và brute đúng (n=50 -> 0.625, n=100 -> 0.71875)


### B. Edge case

- `n = 0`: hai nồi cùng rỗng ngay từ đầu → `0,5` (theo quy ước cùng hết).
- `n = 1, …, 25`: cùng `N = 1` → cùng đáp án `0,625` (kiểm tra phép **chia lên**).
- `n = 26..50` → `N = 2` → `0,625` (như `n = 50`); `n = 51..75` → `N = 3`.
- Sát ngưỡng cắt cụt: `n = 4800` (tính thật) và `n = 4801` (trả `1`).
- `n = 10^9`: phải trả `1` ngay (không đệ quy).

In [21]:
# n = 0
check(0, Fraction(1, 2))
assert soup_brute(0) == Fraction(1, 2)

# chia lên: các n cùng ceil(n/25) cho cùng đáp án
for N in range(1, 12):
    values = {Solution().soupServings(n) for n in range(25 * (N - 1) + 1, 25 * N + 1)}
    assert len(values) == 1, (N, values)
print("mọi n trong cùng khoảng (25(N-1), 25N] cho cùng đáp án: OK với N = 1..11")
assert Solution().soupServings(1) == Solution().soupServings(25) == Solution().soupServings(50) == 0.625

# giá trị đầu dãy chính xác
head = {1: Fraction(5, 8), 25: Fraction(5, 8), 50: Fraction(5, 8), 75: Fraction(21, 32), 100: Fraction(23, 32)}
for n, want in head.items():
    check(n, want)
print("N=3 (n=75):", f_exact(3, 3), "=", float(f_exact(3, 3)))

# sát ngưỡng và rất lớn
for n in (4799, 4800, 4801, 4802, 10 ** 5, 10 ** 9):
    got = Solution().soupServings(n)
    print(f"n = {n:>10d}: {got!r}")
assert Solution().soupServings(4801) == 1 and Solution().soupServings(10 ** 9) == 1
assert Solution().soupServings(4800) < 1

mọi n trong cùng khoảng (25(N-1), 25N] cho cùng đáp án: OK với N = 1..11
N=3 (n=75): 21/32 = 0.65625
n =       4799: 0.999994994425816
n =       4800: 0.999994994425816
n =       4801: 1
n =       4802: 1
n =     100000: 1
n = 1000000000: 1


### C. Phản ví dụ: cài đặt sai phải bị bắt

Mỗi bản sai khác đáp án đúng **vượt dung sai** `1e-5` ít nhất ở một `n`:

In [22]:
def make_bug(floor_units=False, both=0.5, i_first=False, ops=((4, 0), (3, 1), (2, 2), (1, 3)), cutoff=4800):
    def solve(n):
        if n > cutoff:
            return 1.0

        @cache
        def dfs(i, j):
            if i_first:                                  # kiểm tra i<=0 TRƯỚC (bỏ qua trường hợp hòa)
                if i <= 0:
                    return 1.0
                if j <= 0:
                    return 0.0
            else:
                if i <= 0 and j <= 0:
                    return both
                if i <= 0:
                    return 1.0
                if j <= 0:
                    return 0.0
            return 0.25 * sum(dfs(i - da, j - db) for da, db in ops)

        N = n // 25 if floor_units else (n + 24) // 25
        return dfs(N, N)

    return solve


bug_floor = make_bug(floor_units=True)                       # chia sàn thay vì ceil
bug_both_one = make_bug(both=1.0)                            # cùng hết tính là 1
bug_both_zero = make_bug(both=0.0)                           # cùng hết tính là 0
bug_i_first = make_bug(i_first=True)                         # kiểm tra i<=0 trước: hòa thành 1
bug_wrong_ops = make_bug(ops=((4, 0), (3, 1), (2, 2), (0, 4)))   # thao tác 4 sai (0,4) thay vì (1,3)
bug_low_cutoff = make_bug(cutoff=2000)                       # cắt cụt quá sớm


def bug_float_eq(n):                                         # trả lời đúng nhưng test so sánh bằng ==
    return Solution().soupServings(n) + 1e-15


BUGS = [bug_floor, bug_both_one, bug_both_zero, bug_i_first, bug_wrong_ops, bug_low_cutoff]
names = ["bug_floor (n//25)", "bug_both_one", "bug_both_zero", "bug_i_first", "bug_wrong_ops", "bug_low_cutoff"]
for bug, name in zip(BUGS, names):
    bad = next(n for n in list(range(0, 600)) + [2500, 3000, 4000, 4700] if abs(bug(n) - Solution().soupServings(n)) > 1e-5)
    print(f"{name:20s} sai tại n = {bad:5d}: ra {bug(bad):.6f}, đúng {Solution().soupServings(bad):.6f}")

bug_floor (n//25)    sai tại n =     1: ra 0.500000, đúng 0.625000
bug_both_one         sai tại n =     0: ra 1.000000, đúng 0.500000
bug_both_zero        sai tại n =     0: ra 0.000000, đúng 0.500000
bug_i_first          sai tại n =     0: ra 1.000000, đúng 0.500000
bug_wrong_ops        sai tại n =     1: ra 0.500000, đúng 0.625000


bug_low_cutoff       sai tại n =  2500: ra 1.000000, đúng 0.999255


### D. So sánh chính xác bằng `Fraction`, dung sai cho `float`, kiểm chứng cắt cụt

**D1.** Với `n = 0..250`: đệ quy **brute theo mL** (`soup_brute`, chính xác) phải **bằng đúng** bản đơn vị `f_exact(N, N)` (kiểm chứng phép nén đơn vị) và gần các cài đặt `float` trong `1e-9`.

In [23]:
t0 = time.perf_counter()
for n in range(0, 251):
    exact_brute = soup_brute(n)
    N = (n + 24) // 25
    assert exact_brute == f_exact(N, N), n                          # nén đơn vị đúng tuyệt đối
    for name in IMPLS:
        assert close(IMPLS[name](n), float(exact_brute)), (name, n)
print("n = 0..250: brute theo mL == bản đơn vị (Fraction), mọi cài đặt float khớp trong 1e-9; mất", round(time.perf_counter() - t0, 1), "s")

n = 0..250: brute theo mL == bản đơn vị (Fraction), mọi cài đặt float khớp trong 1e-9; mất 0.1 s


**D2.** Với mọi `n = 0..4800` (tức `N = 0..192`): `Fraction` chính xác (bảng đơn vị, hai cách độc lập: đệ quy nhớ và forward) bằng nhau, và mọi cài đặt `float` khớp trong `1e-9`.

In [24]:
t0 = time.perf_counter()
N_MAX = 192
exact_fwd = {}
for N in (1, 2, 3, 7, 19, 64, 100, 150, 192):
    n = N * 25
    assert f_exact(N, N) == soup_forward(n, exact=True), N         # hai cách chính xác độc lập bằng nhau
    exact_fwd[N] = f_exact(N, N)
print("f_exact (nhớ) == forward (Fraction) tại N =", list(exact_fwd), "-> bằng nhau tuyệt đối")

worst = 0.0
for n in list(range(0, 4801, 7)) + [4799, 4800]:
    N = (n + 24) // 25
    want = float(f_exact(N, N))
    worst = max(worst, abs(want - Solution().soupServings(n)), abs(want - SolutionTable().soupServings(n)))
    assert close(Solution().soupServings(n), want) and close(SolutionTable().soupServings(n), want)
print(f"sai số float lớn nhất so với Fraction trên ~690 giá trị n: {worst:.2e}; mất {time.perf_counter() - t0:.1f} s")
assert worst < 1e-12

f_exact (nhớ) == forward (Fraction) tại N = [1, 2, 3, 7, 19, 64, 100, 150, 192] -> bằng nhau tuyệt đối


sai số float lớn nhất so với Fraction trên ~690 giá trị n: 2.22e-16; mất 9.1 s


**D3. Kiểm chứng cắt cụt.** (i) `f(N, N)` **không giảm** theo `N` (tính bằng bảng DP `float` tới `N = 600`); (ii) `1 − f(N, N) < 10^-5` cho mọi `N ≥ 179` (tức `n ≥ 4451`) — vậy cắt tại `n > 4800` (`N > 192`) an toàn; (iii) giá trị `Fraction` chính xác tại `N = 192`.

In [25]:
big = full_table(600)
diag = [big[k][k] for k in range(1, 601)]
assert all(diag[k] >= diag[k - 1] - 1e-15 for k in range(1, 600)), "f(N,N) không đơn điệu"
first_ok = next(k for k in range(1, 601) if 1 - diag[k - 1] < 1e-5)
print("f(N,N) không giảm theo N (N = 1..600): OK")
print("N nhỏ nhất có 1 - f < 1e-5:", first_ok, "-> n >=", (first_ok - 1) * 25 + 1)
assert all(1 - diag[k - 1] < 1e-5 for k in range(first_ok, 601))
print("với mọi N trong [", first_ok, ", 600 ]: 1 - f < 1e-5")
e192 = f_exact(192, 192)
print("1 - f(192,192) =", float(1 - e192), "< 1e-5:", 1 - e192 < Fraction(1, 10 ** 5))
assert 1 - e192 < Fraction(1, 10 ** 5)
# n = 4801 mới là N = 193: cắt cụt trả 1; sai số so với giá trị thật nhỏ hơn dung sai
assert abs(1 - big[193][193]) < 1e-5 and abs(1 - big[600][600]) < 1e-5
print("cắt cụt tại n > 4800 sai lệch tối đa:", 1 - big[193][193])

f(N,N) không giảm theo N (N = 1..600): OK
N nhỏ nhất có 1 - f < 1e-5: 179 -> n >= 4451
với mọi N trong [ 179 , 600 ]: 1 - f < 1e-5
1 - f(192,192) = 5.0055741839972035e-06 < 1e-5: True
cắt cụt tại n > 4800 sai lệch tối đa: 4.6176847512402475e-06


### E. Mô phỏng Monte Carlo (RNG tự dựng có seed)

Ta kiểm tra **thống kê** độc lập với toàn bộ phân tích: mô phỏng trực tiếp quá trình múc súp. Dùng bộ sinh số tự dựng `XorShift64` (seed cố định, nên chạy lại cho cùng kết quả), lấy mẫu **loại bỏ** để chọn đều trong `{0,1,2,3}`. Mỗi ván cho giá trị `1` (A hết trước), `0,5` (hòa) hoặc `0` (B hết trước); trung bình `T` ván hội tụ về đáp án với sai số chuẩn `≤ 0,5/√T`. Ta yêu cầu `|trung bình − đáp án chính xác| ≤ 4σ` với `σ = 0,5/√T`.

In [26]:
class XorShift64:
    MASK = (1 << 64) - 1

    def __init__(self, seed):
        z = (seed + 0x9E3779B97F4A7C15) & self.MASK
        z = ((z ^ (z >> 30)) * 0xBF58476D1CE4E5B9) & self.MASK
        z = ((z ^ (z >> 27)) * 0x94D049BB133111EB) & self.MASK
        z ^= z >> 31
        self.state = z or 0x2545F4914F6CDD1D

    def next_u64(self):
        x = self.state
        x ^= (x << 13) & self.MASK
        x ^= x >> 7
        x ^= (x << 17) & self.MASK
        self.state = x
        return x

    def randrange(self, n):
        limit = (1 << 64) - ((1 << 64) % n)
        while True:
            r = self.next_u64()
            if r < limit:
                return r % n


def simulate(n, trials, seed):
    rng = XorShift64(seed)
    total = 0.0
    for _ in range(trials):
        a = b = n
        while a > 0 and b > 0:
            da, db = OPS[rng.randrange(4)]
            a -= da
            b -= db
        total += 1.0 if (a <= 0 and b > 0) else 0.5 if (a <= 0 and b <= 0) else 0.0
    return total / trials


T = 60000
sigma_max = 0.5 / math.sqrt(T)
print(f"T = {T} ván, sai số chuẩn tối đa σ = {sigma_max:.5f}, ngưỡng 4σ = {4 * sigma_max:.5f}")
for n, seed in [(50, 1), (100, 2), (300, 3), (700, 4), (1500, 5), (3000, 6)]:
    exact = float(Solution().soupServings(n))
    est = simulate(n, T, seed)
    z = (est - exact) / sigma_max
    print(f"n = {n:5d}: đáp án = {exact:.6f}, mô phỏng = {est:.6f}, z <= {z:+.2f}")
    assert abs(est - exact) <= 4 * sigma_max, (n, est, exact)
print("mô phỏng khớp đáp án trong 4σ ở mọi n")

# hai lần chạy cùng seed cho cùng kết quả (tái lập được)
assert simulate(100, 2000, 42) == simulate(100, 2000, 42)

T = 60000 ván, sai số chuẩn tối đa σ = 0.00204, ngưỡng 4σ = 0.00816
n =    50: đáp án = 0.625000, mô phỏng = 0.624108, z <= -0.44
n =   100: đáp án = 0.718750, mô phỏng = 0.714592, z <= -2.04


n =   300: đáp án = 0.852173, mô phỏng = 0.854133, z <= +0.96


n =   700: đáp án = 0.950555, mô phỏng = 0.949292, z <= -0.62


n =  1500: đáp án = 0.992832, mô phỏng = 0.993292, z <= +0.23


n =  3000: đáp án = 0.999753, mô phỏng = 0.999700, z <= -0.03
mô phỏng khớp đáp án trong 4σ ở mọi n


Chú ý: để chắc đáp án trong `10^-5` bằng mô phỏng ta cần `σ ≲ 10^-6`, tức `T ≳ 2,5·10^11` ván — bất khả. Vì vậy mô phỏng **chỉ kiểm tra thống kê** (có đúng cỡ `10^-2`), còn độ chính xác `10^-5` đến từ `Fraction` ở nhóm D.

### F. Giới hạn lớn nhất

`n = 10^9` (và vài giá trị rất lớn khác) phải trả `1` ngay; ca nặng nhất tính thật là `n = 4800` (`N = 192`).

In [27]:
for n in (4800, 4801, 10 ** 6, 10 ** 9):
    t0 = time.perf_counter()
    vals = [IMPLS[name](n) for name in IMPLS]
    dt = time.perf_counter() - t0
    assert max(vals) - min(vals) <= 1e-9
    print(f"n = {n:>10d}: {vals[0]!r:>22} (7 cài đặt khớp trong 1e-9; {dt * 1000:.1f} ms)")
assert Solution().soupServings(10 ** 9) == 1.0
print("giới hạn lớn nhất OK")

n =       4800:      0.999994994425816 (7 cài đặt khớp trong 1e-9; 88.1 ms)
n =       4801:                    1.0 (7 cài đặt khớp trong 1e-9; 0.0 ms)
n =    1000000:                    1.0 (7 cài đặt khớp trong 1e-9; 0.0 ms)
n = 1000000000:                    1.0 (7 cài đặt khớp trong 1e-9; 0.0 ms)
giới hạn lớn nhất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chia sàn `n // 25` thay vì chia lên

`n = 51` mL cần **3** đơn vị (đủ 3 lần múc 25 mL mới hết); `51 // 25 = 2` sai (và cho `0,625` thay vì `0,65625`). Lưu ý với một số `n` (như `n = 26`) hai đáp án tình cờ trùng nhau vì `f(1,1) = f(2,2)`, nên phải thử nhiều `n`.

In [28]:
for n in (26, 51, 60, 76, 101, 130):
    print(f"n={n:3d}: bug_floor = {bug_floor(n):.5f} | đúng = {Solution().soupServings(n):.5f}")
assert abs(bug_floor(51) - Solution().soupServings(51)) > 1e-5

n= 26: bug_floor = 0.62500 | đúng = 0.62500
n= 51: bug_floor = 0.62500 | đúng = 0.65625
n= 60: bug_floor = 0.62500 | đúng = 0.65625
n= 76: bug_floor = 0.65625 | đúng = 0.71875
n=101: bug_floor = 0.71875 | đúng = 0.74219
n=130: bug_floor = 0.74219 | đúng = 0.75781


**Vì sao sai:** lượng dư (1–24 mL) vẫn cần thêm một lần múc để hết. **Đúng:** `(n + 24) // 25` (hoặc `math.ceil(n / 25)`).

### Lỗi 2: giá trị cơ sở hòa sai (`1` hoặc `0` thay vì `0,5`)

Đề: "xác suất A hết trước **cộng một nửa** xác suất hai nồi cùng hết". Cả hai cùng hết phải ra `0,5`.

In [29]:
for n in (50, 100):
    print(f"n={n}: hòa=1 -> {bug_both_one(n):.5f} | hòa=0 -> {bug_both_zero(n):.5f} | đúng -> {Solution().soupServings(n):.5f}")
assert bug_both_one(50) == 0.75 and bug_both_zero(50) == 0.5

n=50: hòa=1 -> 0.75000 | hòa=0 -> 0.50000 | đúng -> 0.62500
n=100: hòa=1 -> 0.81250 | hòa=0 -> 0.62500 | đúng -> 0.71875


**Vì sao sai:** ví dụ 1 có đúng một trong bốn nhánh là hòa, đóng góp `¼ × 0,5 = 0,125`; bản sai cho `¼ × 1` hay `¼ × 0`. **Đúng:** `return 0.5`.

### Lỗi 3: kiểm tra điều kiện sai thứ tự

Nếu kiểm tra `if i <= 0: return 1` **trước** khi xét trường hợp cả hai `≤ 0`, thì hòa bị tính thành "A hết trước" (`1`).

In [30]:
print("n=50 sai thứ tự:", bug_i_first(50), "| đúng:", Solution().soupServings(50))
assert bug_i_first(50) == 0.75

n=50 sai thứ tự: 0.75 | đúng: 0.625


**Vì sao sai:** trường hợp cả hai `≤ 0` đã thoả `i ≤ 0`. **Đúng:** xét "cả hai" trước, rồi mới từng nồi.

### Lỗi 4: quên cắt cụt cho `n` lớn

Không có dòng `n > 4800 → 1`, đệ quy có nhớ đi sâu `N = n/25` tầng: với `n = 10^5` (`N = 4000`) đã `RecursionError`; với `n = 10^9` còn tốn bộ nhớ khổng lồ.

In [31]:
def soup_no_cutoff(n):
    @cache
    def dfs(i, j):
        if i <= 0 and j <= 0:
            return 0.5
        if i <= 0:
            return 1.0
        if j <= 0:
            return 0.0
        return 0.25 * (dfs(i - 4, j) + dfs(i - 3, j - 1) + dfs(i - 2, j - 2) + dfs(i - 1, j - 3))

    return dfs((n + 24) // 25, (n + 24) // 25)


old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    soup_no_cutoff(100000)
    print("không lỗi (bất ngờ)")
except RecursionError:
    print("soup_no_cutoff(100000) -> RecursionError (N = 4000 tầng đệ quy)")
finally:
    sys.setrecursionlimit(old)
print("với cắt cụt:", Solution().soupServings(100000), "(tức thì)")

soup_no_cutoff(100000) -> RecursionError (N = 4000 tầng đệ quy)
với cắt cụt: 1 (tức thì)


**Vì sao sai:** độ sâu và số trạng thái phụ thuộc `n`. **Đúng:** nhận ra đáp án tiến về `1` và cắt cụt (`n > 4800`).

### Lỗi 5: sai thao tác hoặc ngưỡng cắt cụt quá nhỏ

Gõ nhầm thao tác (như `(0,4)` thay `(1,3)`) hoặc chọn ngưỡng cắt cụt quá thấp (như `2000`) cho đáp án lệch hơn `10^-5`.

In [32]:
n = 100
print("thao tác sai:", bug_wrong_ops(n), "| đúng:", Solution().soupServings(n))
n = 3000
print("cắt cụt 2000 tại n=3000:", bug_low_cutoff(n), "| đúng:", Solution().soupServings(n), "| lệch", abs(bug_low_cutoff(n) - Solution().soupServings(n)))
assert abs(bug_low_cutoff(3000) - Solution().soupServings(3000)) > 1e-5

thao tác sai: 0.59375 | đúng: 0.71875
cắt cụt 2000 tại n=3000: 1.0 | đúng: 0.9997529725570642 | lệch 0.0002470274429358099


**Vì sao sai:** thao tác (0,4) múc 100 mL từ B và 0 từ A — **không tồn tại** (đề nhấn mạnh điều đó). Ngưỡng `2000` chưa đủ để `1 − f < 10^-5` (cần `n ≥ 4451`). **Đúng:** `4800` (hoặc lớn hơn).

### Lỗi 6: so sánh số thực bằng `==` trong test

Kết quả `float` thường lệch chữ số cuối; `assert result == 0.71875` có thể thất bại dù đúng.

In [33]:
r = bug_float_eq(100)
print("r =", repr(r), "| r == 0.71875 ?", r == 0.71875, "| |r - 0.71875| <= 1e-9 ?", abs(r - 0.71875) <= 1e-9)
assert r != 0.71875 and abs(r - 0.71875) <= 1e-9

r = 0.718750000000001 | r == 0.71875 ? False | |r - 0.71875| <= 1e-9 ? True


**Vì sao sai:** sai số làm tròn `~10^-16`. **Đúng:** `abs(a - b) <= eps` hoặc `math.isclose`; đề cũng chỉ yêu cầu `10^-5`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "xác suất của quá trình ngẫu nhiên có trạng thái hữu hạn, mỗi bước chọn đều một thao tác, dừng khi chạm điều kiện" → **truy hồi xác suất + quy hoạch động / nhớ kết quả** (luật xác suất toàn phần).

**Mẹo:**

1. **Giảm đơn vị** theo ước chung lớn nhất của các lượng thay đổi (ở đây 25) để thu nhỏ không gian trạng thái.
2. Viết rõ **giá trị cơ sở** theo từng trường hợp dừng và **thứ tự** kiểm tra.
3. Khi tham số lớn mà đáp án **hội tụ** (tiến tới `0` hoặc `1`), cắt cụt ở ngưỡng mà sai số nhỏ hơn dung sai; **kiểm chứng ngưỡng bằng số học chính xác**.
4. Dùng `Fraction` làm **oracle** cho bài số thực; dùng `float` có dung sai trong test, không `==`.
5. Kiểm tra thống kê bằng mô phỏng có seed (z-score) như một lớp bảo vệ độc lập.
6. Hai hướng tính (ngược `f(i,j)` và xuôi phân phối theo lượt) cho kết quả bằng nhau — dùng để cross-check.

**Họ bài cùng pattern:** Knight Probability in Chessboard (688), New 21 Game (837), Probability of a Two Boxes Having The Same Number of Distinct Balls (1467), Dice Roll Simulation (1223), Number of Ways to Stay in the Same Place After Some Steps (1269).

## 15. Biến thể

1. **Xác suất tách riêng:** `P(A hết trước)`, `P(hòa)`, `P(B hết trước)` (ba số cộng lại bằng `1`); đáp án gốc là `P_A + P_hòa / 2`.
2. **Kỳ vọng số lượt** cho tới khi có nồi cạn (bài tập 5).
3. **Xác suất các thao tác khác nhau** (không đều `¼`).
4. **Ba nồi súp** hoặc nồi B được thêm thao tác `(0, 100)`: thay đổi `OPS`.
5. **Số lượt tối đa cố định `T`:** xác suất tại thời điểm `T`.

Cài đặt biến thể 1 bằng `Fraction` và kiểm tra tổng bằng `1`, và công thức `P_A + P_hòa/2 = đáp án`:

In [34]:
@cache
def triple(i, j):
    """(P(A hết trước), P(hòa), P(B hết trước)) chính xác, khi A còn i đơn vị, B còn j đơn vị."""
    if i <= 0 and j <= 0:
        return (Fraction(0), Fraction(1), Fraction(0))
    if i <= 0:
        return (Fraction(1), Fraction(0), Fraction(0))
    if j <= 0:
        return (Fraction(0), Fraction(0), Fraction(1))
    parts = [triple(i - 4, j), triple(i - 3, j - 1), triple(i - 2, j - 2), triple(i - 1, j - 3)]
    return tuple(sum(p[k] for p in parts) / 4 for k in range(3))


for N in (1, 2, 4, 8, 20, 60):
    pa, pt, pb = triple(N, N)
    assert pa + pt + pb == 1
    assert pa + pt / 2 == f_exact(N, N)
    print(f"N={N:2d} (n={N * 25:4d}): P(A trước)={float(pa):.6f}  P(hòa)={float(pt):.6f}  P(B trước)={float(pb):.6f}")
print("P(A trước) + P(hòa)/2 = đáp án, và ba xác suất cộng lại bằng 1 (chính xác bằng Fraction)")

N= 1 (n=  25): P(A trước)=0.250000  P(hòa)=0.750000  P(B trước)=0.000000
N= 2 (n=  50): P(A trước)=0.500000  P(hòa)=0.250000  P(B trước)=0.250000
N= 4 (n= 100): P(A trước)=0.625000  P(hòa)=0.187500  P(B trước)=0.187500
N= 8 (n= 200): P(A trước)=0.742188  P(hòa)=0.109375  P(B trước)=0.148438
N=20 (n= 500): P(A trước)=0.897341  P(hòa)=0.038007  P(B trước)=0.064652
N=60 (n=1500): P(A trước)=0.991408  P(hòa)=0.002848  P(B trước)=0.005744
P(A trước) + P(hòa)/2 = đáp án, và ba xác suất cộng lại bằng 1 (chính xác bằng Fraction)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy thuần theo mL | Truy hồi trực tiếp, không nhớ | `O(4^N)` | `O(N)` | Oracle (`n ≤ 250`); kiểm chứng nén đơn vị |
| 2. Đệ quy nhớ + nén + cắt cụt | `f(i, j)` đơn vị 25 mL, `@cache`, `n > 4800 → 1` | `O(N²)` | `O(N²)` | **Đáp án chính**; ngắn gọn |
| 3. DP bảng dưới lên | Điền `f[i][j]` theo thứ tự tăng | `O(N²)` | `O(N²)` | Không đệ quy; cho mọi `N' ≤ N` |
| 4. Forward (lan truyền phân phối) | Phân phối xác suất theo từng lượt, cộng khi dừng | `O(N³)` xấu nhất | `O(N²)` | Cross-check độc lập; mở rộng sang kỳ vọng số lượt |

Cả bốn khớp nhau và khớp `Fraction` chính xác (mục 12D); mô phỏng Monte Carlo khớp trong 4σ (mục 12E).

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [35]:
print((0 + 24) // 25, (1 + 24) // 25, (25 + 24) // 25, (26 + 24) // 25, (100 + 24) // 25)
print(Solution().soupServings(0), Solution().soupServings(1), Solution().soupServings(25), Solution().soupServings(10 ** 9))

0 1 1 2 4
0.5 0.625 0.625 1


### Bài 2 (tính tay)

- (a) Tính `f(1, 1)` (tức `n = 25`) bằng công thức truy hồi và giá trị cơ sở: bốn nhánh cho gì?
- (b) Tính `f(2, 1)`: A còn 2 đơn vị, B còn 1 đơn vị.
- (c) Giải thích vì sao `n = 1` và `n = 25` cho cùng đáp án.

Tự giải rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện đệ quy có nhớ:

In [36]:
def soup_todo(n):
    @cache
    def dfs(i, j):
        # TODO 1: trường hợp cơ sở (cả hai <= 0 -> 0.5; i <= 0 -> 1; j <= 0 -> 0), đúng THỨ TỰ
        # TODO 2: trả 0.25 * tổng 4 nhánh (i-4,j), (i-3,j-1), (i-2,j-2), (i-1,j-3)
        return None

    return 1.0 if n > 4800 else dfs((n + 24) // 25, (n + 24) // 25)


# Khi làm xong, bỏ comment để kiểm tra:
# assert abs(soup_todo(50) - 0.625) < 1e-9 and abs(soup_todo(100) - 0.71875) < 1e-9

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **DP bảng** (không đệ quy):

In [37]:
class MySolution:
    def soupServings(self, n):
        pass            # TODO


# Khi làm xong:
# assert abs(MySolution().soupServings(50) - 0.625) < 1e-9
# assert abs(MySolution().soupServings(100) - 0.71875) < 1e-9
# assert MySolution().soupServings(10 ** 9) == 1.0

### Bài 5 (mini challenge): kỳ vọng số lượt

Viết `expected_turns(n)` trả về (bằng `Fraction`) **kỳ vọng số lượt** cho tới khi một nồi (hoặc cả hai) cạn. Gợi ý: `E(i, j) = 1 + ¼ · Σ E(i', j')` trên các nhánh còn tiếp tục (`i' > 0 và j' > 0`), `E = 0` khi đã dừng. Ví dụ `n = 25` (một đơn vị): mọi thao tác đều làm A cạn nên `E = 1`. Kiểm chứng bằng mô phỏng.

In [38]:
def expected_turns(n):
    pass            # TODO


# Khi làm xong:
# assert expected_turns(25) == 1
# assert expected_turns(0) == 0

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `(n + 24) // 25` là phép chia lên.
- Mức 2: `0 → 0`, `1 → 1`, `25 → 1`, `26 → 2`, `100 → 4`.
- Mức 3: `soupServings`: `n = 0 → 0.5`; `n = 1, 25 → 0.625`; `n = 10^9 → 1`.

**Bài 2**

- Mức 1: đổi mọi thứ thành đơn vị 25 mL.
- Mức 2: `f(1,1)`: bốn nhánh `(−3,1), (−2,0), (−1,−1), (0,−2)` → `1, 0.5, 0.5, 0.5`.
- Mức 3: `f(1,1) = ¼·(1 + 0,5 + 0,5 + 0,5) = 0,625`. `f(2,1)`: các nhánh `(−2,1), (−1,0), (0,−1), (1,−2)` → `1, 0.5, 0.5, 0` → `0,5`. Cùng `ceil`.

**Bài 3**

- Mức 1: ba `if` đầu hàm.
- Mức 2: `if i <= 0 and j <= 0: return 0.5` phải đứng đầu.
- Mức 3: `return 0.25 * (dfs(i-4,j) + dfs(i-3,j-1) + dfs(i-2,j-2) + dfs(i-1,j-3))`.

**Bài 4**

- Mức 1: `N = (n + 24) // 25`, bảng `(N+1) x (N+1)`.
- Mức 2: hàm phụ `get(i, j)` trả giá trị cơ sở khi `i <= 0` hoặc `j <= 0`.
- Mức 3: vòng `for i in 1..N: for j in 1..N`; nhớ `n > 4800 → 1`.

**Bài 5**

- Mức 1: `E(i, j) = 1 + ¼ Σ E(nhánh còn tiếp tục)`.
- Mức 2: nhánh đã dừng (`i' ≤ 0` hoặc `j' ≤ 0`) đóng góp `0`.
- Mức 3: dùng `Fraction`; không cắt cụt (kỳ vọng số lượt tăng theo `n`; cho `n` nhỏ). Kiểm chứng bằng `soup_forward`: `E = Σ_t P(chưa dừng sau t lượt)`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [39]:
assert [(n + 24) // 25 for n in (0, 1, 25, 26, 100)] == [0, 1, 1, 2, 4]
assert (Solution().soupServings(0), Solution().soupServings(1), Solution().soupServings(25), Solution().soupServings(10 ** 9)) == (0.5, 0.625, 0.625, 1)
print("Bài 1 OK")

Bài 1 OK


### Bài 2

(a) `f(1,1) = ¼·(1 + 0,5 + 0,5 + 0,5) = 0,625`. (b) `f(2,1) = ¼·(1 + 0,5 + 0,5 + 0) = 0,5`. (c) `n = 1` và `n = 25` đều cần **1 đơn vị**: mọi thao tác múc ít nhất 25 mL từ A nên A hết ngay lượt đầu; B hết khi nào tuỳ thao tác.

In [40]:
assert f_exact(1, 1) == Fraction(5, 8)
assert f_exact(2, 1) == Fraction(1, 2)
assert Solution().soupServings(1) == Solution().soupServings(25) == 0.625
assert soup_brute(1) == soup_brute(25) == Fraction(5, 8)
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [41]:
def soup_answer(n):
    @cache
    def dfs(i, j):
        if i <= 0 and j <= 0:
            return 0.5
        if i <= 0:
            return 1.0
        if j <= 0:
            return 0.0
        return 0.25 * (dfs(i - 4, j) + dfs(i - 3, j - 1) + dfs(i - 2, j - 2) + dfs(i - 1, j - 3))

    return 1.0 if n > 4800 else dfs((n + 24) // 25, (n + 24) // 25)


assert abs(soup_answer(50) - 0.625) < 1e-9 and abs(soup_answer(100) - 0.71875) < 1e-9
for n in range(0, 800, 11):
    assert abs(soup_answer(n) - float(f_exact((n + 24) // 25, (n + 24) // 25))) < 1e-12
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [42]:
class MySolutionAnswer:
    def soupServings(self, n):
        if n > 4800:
            return 1.0
        N = (n + 24) // 25
        if N == 0:
            return 0.5
        f = [[0.0] * (N + 1) for _ in range(N + 1)]

        def get(i, j):
            if i <= 0 and j <= 0:
                return 0.5
            if i <= 0:
                return 1.0
            if j <= 0:
                return 0.0
            return f[i][j]

        for i in range(1, N + 1):
            for j in range(1, N + 1):
                f[i][j] = 0.25 * (get(i - 4, j) + get(i - 3, j - 1) + get(i - 2, j - 2) + get(i - 1, j - 3))
        return f[N][N]


assert abs(MySolutionAnswer().soupServings(50) - 0.625) < 1e-9
assert abs(MySolutionAnswer().soupServings(100) - 0.71875) < 1e-9
assert MySolutionAnswer().soupServings(10 ** 9) == 1.0
for n in range(0, 5000, 13):
    assert abs(MySolutionAnswer().soupServings(n) - Solution().soupServings(n)) < 1e-12
print("Bài 4 OK")

Bài 4 OK


### Bài 5

`E(i, j)` = kỳ vọng số lượt tính từ trạng thái `(i, j)` (còn chưa dừng); mỗi lượt tốn `1`, rồi tiếp tục với các nhánh chưa dừng.

In [43]:
def expected_turns_answer(n):
    @cache
    def E(i, j):
        if i <= 0 or j <= 0:
            return Fraction(0)
        total = Fraction(0)
        for da, db in ((4, 0), (3, 1), (2, 2), (1, 3)):
            total += E(i - da, j - db)
        return 1 + total / 4

    N = (n + 24) // 25
    return E(N, N)


assert expected_turns_answer(25) == 1 and expected_turns_answer(0) == 0
print("E(n=50) =", expected_turns_answer(50), "| E(n=100) =", expected_turns_answer(100), "| E(n=500) =", float(expected_turns_answer(500)))

# kiểm chứng độc lập 1: E = tổng_t P(chưa dừng sau t lượt) bằng phân phối forward chính xác
def expected_turns_forward(n):
    N = (n + 24) // 25
    if N == 0:
        return Fraction(0)                              # đã cạn ngay từ đầu: 0 lượt
    dist = {(N, N): Fraction(1)}
    total = Fraction(0)
    while dist:
        total += sum(dist.values())                      # mỗi trạng thái chưa dừng đóng góp một lượt nữa
        new = {}
        for (i, j), p in dist.items():
            for da, db in ((4, 0), (3, 1), (2, 2), (1, 3)):
                i2, j2 = i - da, j - db
                if i2 > 0 and j2 > 0:
                    new[(i2, j2)] = new.get((i2, j2), 0) + p / 4
        dist = new
    return total


for n in (0, 1, 25, 50, 100, 250, 500):
    assert expected_turns_answer(n) == expected_turns_forward(n), n

# kiểm chứng độc lập 2: mô phỏng Monte Carlo có seed
def simulate_turns(n, trials, seed):
    rng = XorShift64(seed)
    total = 0
    for _ in range(trials):
        a = b = n
        t = 0
        while a > 0 and b > 0:
            da, db = OPS[rng.randrange(4)]
            a -= da
            b -= db
            t += 1
        total += t
    return total / trials


for n, seed in [(100, 1), (500, 2)]:
    est = simulate_turns(n, 40000, seed)
    exact = float(expected_turns_answer(n))
    print(f"n = {n}: kỳ vọng chính xác {exact:.4f}, mô phỏng {est:.4f}")
    assert abs(est - exact) < 0.05
print("Bài 5 OK")

E(n=50) = 1 | E(n=100) = 7/4 | E(n=500) = 8.27853775024414
n = 100: kỳ vọng chính xác 1.7500, mô phỏng 1.7507


n = 500: kỳ vọng chính xác 8.2785, mô phỏng 8.2847
Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (xác suất trạng thái hữu hạn, đệ quy có nhớ):**

```python
class Solution:
    def soupServings(self, n: int) -> float:
        @cache
        def dfs(i, j):                         # i, j: số đơn vị 25 mL còn lại của A, B
            if i <= 0 and j <= 0:              # cùng hết
                return 0.5
            if i <= 0:                         # A hết trước
                return 1
            if j <= 0:                         # B hết trước
                return 0
            return 0.25 * (dfs(i - 4, j) + dfs(i - 3, j - 1) + dfs(i - 2, j - 2) + dfs(i - 1, j - 3))

        return 1 if n > 4800 else dfs((n + 24) // 25, (n + 24) // 25)
```

**Checklist trước khi submit:**

- [ ] Đổi sang đơn vị 25 mL bằng phép chia **lên** `(n + 24) // 25`?
- [ ] Giá trị cơ sở: cả hai `≤ 0 → 0.5`, `i ≤ 0 → 1`, `j ≤ 0 → 0`, kiểm tra **cả hai trước**?
- [ ] Bốn thao tác đúng `(4,0), (3,1), (2,2), (1,3)` (không có `(0,4)`), xác suất `0.25` mỗi nhánh?
- [ ] Có cắt cụt `n > 4800 → 1` (tránh `RecursionError`, độ phức tạp `O(N²)`)?
- [ ] Khi tự test số thực: dùng dung sai (`abs(a-b) <= 1e-9`), không `==`; dùng `Fraction` làm chuẩn?
- [ ] Đã thử `n = 0 → 0.5`, `n = 50 → 0.625`, `n = 100 → 0.71875`, `n = 10^9 → 1`?
- [ ] Đúng tên `class Solution`, `soupServings(self, n)`, không in debug.